In [5]:
from datetime import datetime
from pathlib import Path
import joblib
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import rcParams
import functools
# from logger.logger import get_logger, setup_logging
import optuna
from logger.logger import get_logger, setup_logging
from model.Granger_causalFormer import PredictModel
from data_loader import TimeSeriesDataloader

# from util import read_json
from train_causalformer import CausalFormerTrainer

In [6]:

# 参数设置
# P = 5           # 时间序列的数量
# T = 1000        # 总时间点
LAG = 2         # 真实的 VAR 滞后
SPARSITY = 0.4  # 格兰杰因果矩阵的稀疏度
BETA_VALUE = 0.8# 系数值
SD = 0.1        # 噪声的标准差
DATA_SEED = 42  # 用于可重复性的随机种子
INPUT_WINDOW = 20 # 输入序列长度 (输入窗口)
FEATURE_DIM = 1 # 每个时间序列在每个时间点上的特征数量
OUTPUT_DIM = 1  # 每个时间序列在每个预测时间步上输出的目标数量
OUTPUT_WINDOW = 1 # 预测下一个时间步


# --- 训练和 Optuna 参数 ---
EPOCHS = 10
BATCH_SIZE = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_TRIALS = 50 # Optuna 的试验次数
STUDY_NAME = "optuna"


# --- 1. 生成并预处理数据 ---
# X_np, _, GC_true_np = simulate_var(p=P, T=T, lag=LAG, sparsity=SPARSITY,
#                                    beta_value=BETA_VALUE, sd=SD, seed=DATA_SEED)

# --- 1. 得到数据并预处理 ---
data_path = 'data/fMRI/timeseries9.csv'
true_gc_path = 'data/fMRI/sim9_gt_processed.csv'

timeseriesDataLoader = TimeSeriesDataloader(data_dir=data_path, gc_dir=true_gc_path, batch_size=BATCH_SIZE, 
                                            DATA_SEED=DATA_SEED, input_window=INPUT_WINDOW, output_window=OUTPUT_WINDOW)
GC_true_np = timeseriesDataLoader.get_true_granger() # 得到真实的格兰杰因果矩阵
train_loader, val_loader, test_loader = timeseriesDataLoader.split_sampler() # 得到训练集、验证集和测试集的数据加载器

In [7]:
input_window = INPUT_WINDOW # 输入序列长度
output_window = OUTPUT_WINDOW # 固定预测下一步
d_model = 32      # QK 嵌入维度
n_head = 4         # 注意力头数
n_layers = 3                     # Encoder 层数
ffn_hidden = 64# FFN 隐藏层维度
dropout = 0.1               # Dropout
tau = 1               # Softmax 温度


# GrangerTCN 参数
tcn_layers = 2                 # TCN 块数
tcn_channels = 32 # TCN 通道数
tcn_kernel_size = 3 # TCN 核大小
tcn_dropout = 0          # TCN Dropout
# tcn_channel_list = [tcn_channels] * tcn_layers

# 近端梯度下降和稀疏性参数
criterion =  nn.MSELoss()
lr = 1e-4     # 学习率
lambda_reg = 1e-5
penalty_type = 'GL'
alpha_gsgl = 0.5 # 仅在GSGL的情况下才有意义，负责控制GSGL内部组稀疏和组内稀疏的平衡

In [8]:
config = {
    'data_loader': {
        'args': {
            'input_window': INPUT_WINDOW,
            'output_window': OUTPUT_WINDOW,
            'feature_dim': FEATURE_DIM,
            'output_dim': OUTPUT_DIM,
            'series_num': 5
        }
    },
    'device': DEVICE.type # 传递设备类型
}

model = PredictModel(config=config,
                        d_model=d_model,
                        n_head=n_head,
                        n_layers=n_layers,
                        tcn_channels=tcn_channels,
                        tcn_kernel_size=tcn_kernel_size,
                        tcn_dropout=tcn_dropout,
                        ffn_hidden=ffn_hidden,
                        drop_prob=dropout,
                        tau=tau).to(DEVICE)

In [21]:
first_layer_param = model.encoder.layers[0].attention.tcn_processors[0].network_layers[0].conv1.weight
print(first_layer_param)
'''
model.encoder.layers[0]：访问模型的第一层编码器层。
.attention：访问该编码器层中的注意力机制。
.tcn_processor：访问注意力机制中的时间卷积网络（TCN）处理器。
.network_layers[0]：访问TCN处理器中的第一个时间块（TemporalBlock）。
.conv1：访问该时间块中的第一个卷积层。
.weight：获取该卷积层的权重参数。
'''


Parameter containing:
tensor([[[-0.3073,  0.5287, -0.1265],
         [-0.3421,  0.1300,  0.0563],
         [ 0.5979,  0.5162,  1.0792],
         [-0.1203,  0.2350, -0.2675]],

        [[-0.0791, -0.9305,  1.3409],
         [-0.9772,  0.5703,  0.0116],
         [ 0.3075, -0.9836, -0.5556],
         [ 0.5144, -0.6756,  0.5994]],

        [[ 0.4872,  0.8090,  0.4819],
         [ 0.3398, -0.4389, -0.4735],
         [-0.1189, -0.6588, -0.3300],
         [ 0.5050,  0.4196, -0.0113]],

        [[ 0.3737, -0.1321, -0.7909],
         [ 0.6992, -0.0145,  0.2050],
         [ 0.3163,  0.1906, -0.2413],
         [ 0.0282,  0.0619, -0.0702]],

        [[-0.1211, -0.4827,  0.1874],
         [-0.4445, -0.1316, -0.1930],
         [-0.5642, -0.1317,  0.3478],
         [-0.4166,  0.0157,  0.7637]],

        [[ 0.5700,  0.2378,  0.1151],
         [ 0.0352, -0.1300, -0.1629],
         [ 0.7050,  0.4810, -0.1773],
         [-0.3710,  0.3913,  0.0896]],

        [[-0.0232,  0.7561,  0.1844],
         [ 0.088

'\nmodel.encoder.layers[0]：访问模型的第一层编码器层。\n.attention：访问该编码器层中的注意力机制。\n.tcn_processor：访问注意力机制中的时间卷积网络（TCN）处理器。\n.network_layers[0]：访问TCN处理器中的第一个时间块（TemporalBlock）。\n.conv1：访问该时间块中的第一个卷积层。\n.weight：获取该卷积层的权重参数。\n'

In [ ]:
print(model)

PredictModel(
  (encoder): Encoder(
    (emb): Embedding(
      (feature_emb): Linear(in_features=20, out_features=32, bias=True)
      (norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
      (drop_out): Dropout(p=0.1, inplace=False)
    )
    (layers): ModuleList(
      (0-2): 3 x EncoderLayer(
        (attention): MultiHeadAttention(
          (Wq): Linear(in_features=32, out_features=32, bias=True)
          (Wk): Linear(in_features=32, out_features=32, bias=True)
          (tcn_processor): GrangerTCN(
            (network_layers): ModuleList(
              (0): TemporalBlock(
                (conv1): Conv1d(5, 32, kernel_size=(3,), stride=(1,), padding=(2,))
                (chomp1): Chomp1d()
                (relu1): ReLU()
                (dropout1): Dropout(p=0, inplace=False)
                (conv2): Conv1d(32, 32, kernel_size=(3,), stride=(1,), padding=(2,))
                (chomp2): Chomp1d()
                (relu2): ReLU()
                (dropout2): Dropout(p=0, 

In [ ]:
print("Granger weights param shape:", first_layer_param.shape)
'''
out_channels = 32：这个卷积层的输出通道数为32，意味着卷积操作后会生成32个特征图。
in_channels = 5：输入数据的通道数为5，这意味着输入数据有5个通道。
kernel_size = 3：卷积核的大小为3，表示卷积操作的窗口大小为3。
torch.Size([32, 5, 3])
'''
print("Granger weights param shape:", first_layer_param)

Granger weights param shape: torch.Size([32, 5, 3])
Granger weights param shape: Parameter containing:
tensor([[[-0.2530,  0.4172,  0.5188],
         [-0.0612,  0.6695, -0.4646],
         [-0.0133,  0.2266,  0.0644],
         [-0.1334, -0.4884, -0.1816],
         [ 0.2427, -0.1411, -0.1482]],

        [[ 0.5600, -0.0409,  0.7606],
         [-0.0734,  0.4207,  0.0021],
         [-0.3310, -0.3349,  0.4968],
         [ 0.4210,  0.8536, -0.4879],
         [ 0.0171,  0.1970,  0.2422]],

        [[ 0.5081,  0.0331,  0.4832],
         [-0.4520,  0.5049, -0.8724],
         [-0.1289,  0.2280,  0.4367],
         [ 0.0632, -0.6386,  0.2503],
         [ 0.4140,  0.3995,  0.6157]],

        [[-0.2228, -0.5752,  0.3996],
         [-0.1910,  0.5426,  0.5123],
         [-0.3853, -0.0104, -0.6864],
         [ 0.1839,  0.0378, -0.0210],
         [-0.4104, -0.5190, -0.2559]],

        [[ 0.6362, -0.5346, -0.3513],
         [ 0.3043,  0.0724,  0.3543],
         [ 0.4205,  0.0606,  0.0500],
         [-0.35

In [ ]:
for name, param in model.named_parameters():
    if param is first_layer_param:
        # w_tilde = param.data - lr * param.grad  # 梯度下降公式
        # lambda_gamma = lr * lambda_reg #  是正则化参数，在近端操作中控制正则化的强度。
        # w_new = torch.zeros_like(w_tilde)  # 初始化新的权重张量
        print("w_tilde shape:", param.shape)
        # for j in range(w_tilde.shape[1]): # 遍历输入特征
        #     w_new[:, j, :] = prox_group_lasso(w_tilde[:, j, :], lambda_gamma)

w_tilde shape: torch.Size([32, 5, 3])


In [ ]:
def load_model(config, best_params, device):
    """
    根据最佳参数加载模型
    """
    # 从最佳参数中提取模型参数
    d_model = best_params['d_model']
    n_head = best_params['n_head']
    n_layers = best_params['n_layers']
    ffn_hidden = best_params['ffn_hidden']
    dropout = best_params['dropout']
    tau = best_params['tau']
    
    # GrangerTCN 参数
    tcn_layers = best_params['tcn_layers']
    tcn_channels = best_params['tcn_channels']
    tcn_kernel_size = best_params['tcn_kernel_size']
    tcn_dropout = best_params['tcn_dropout']
    tcn_channel_list = [tcn_channels] * tcn_layers
    
    # 创建并返回模型
    model = PredictModel(
        config=config,
        d_model=d_model,
        n_head=n_head,
        n_layers=n_layers,
        tcn_channels=tcn_channel_list,
        tcn_kernel_size=tcn_kernel_size,
        tcn_dropout=tcn_dropout,
        ffn_hidden=ffn_hidden,
        drop_prob=dropout,
        tau=tau
    ).to(device)
    
    return model


In [ ]:
best_params_file = "saved/05-09_15-02-13/best_params.pkl"
best_params = joblib.load(best_params_file)

model = load_model(config, best_params, DEVICE) # 构建模型

In [ ]:
GC = model.get_GC(threshold=True, ignore_lag=True)

In [ ]:
print("Granger weights:", GC)
print("Granger weights param shape:", GC.shape)

Granger weights: tensor([[1, 1, 1, 1, 1]], device='cuda:0', dtype=torch.int32)
Granger weights param shape: torch.Size([1, 5])


lr_list: [0.0001, 0.0001, 0.0001, 0.0001, 0.0001]
